# Stage 2 — Simulate adaptation with `SimulationEngine` (recommended path)

This notebook is the post-Phase-3 replacement for `2_simulate_adaptation.ipynb`:
it drives the unified **`SimulationEngine`** with the validated **`SEURule`**
(DYNAMO-M Subjective Expected Utility) instead of the deprecated legacy
`ABMSimulator`. Swap in `ThresholdRule` to reproduce the legacy behaviour
bit-for-bit.

**Inputs**: the stage-1 lookup table (`lookup_table_<site>_<event_set>.nc`,
produced by `1_create_lookup_table.ipynb`) and a per-year SLR trajectory.

*(HYG.4, 2026-07-09 — see `docs/20260709_proposed_development_architecture_steps.md`.)*

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from floodadapt_abm import (
    SimulationEngine,
    CouplingConfig,
    DecisionConfig,
    SEURule,
    ThresholdRule,
)

## 1. Configuration

Point `LOOKUP_TABLE` at your stage-1 output. The SLR trajectory should have one
value per simulated year, in the **same unit as the lookup table's `slr` axis**
(feet for the Charleston case). If you have a FloodAdapt database available you
can derive it from a named projection via `fa.interp_slr(...)`; otherwise a
linear ramp is used below.

In [ ]:
LOOKUP_TABLE = "lookup_table_charleston_beta_release_ABM_test_set.nc"  # <- adjust

START_YEAR = 2020
N_YEARS = 30
NO_SEQ = 10          # Monte-Carlo sequences
SEED = 42

ds = xr.open_dataset(LOOKUP_TABLE)
print(ds.sizes)

# --- SLR trajectory -------------------------------------------------------
# Preferred (needs a FloodAdapt database):
#   import flood_adapt.api as fa
#   slr_values = np.array([
#       fa.interp_slr(slr_scenario="NOAA High", year=START_YEAR + t)
#       for t in range(N_YEARS)
#   ])
# Fallback: linear ramp across the lookup table's SLR grid.
slr_values = np.linspace(float(ds.slr.min()), float(ds.slr.max()), N_YEARS)
years = START_YEAR + np.arange(N_YEARS)

## 2. Build the engine and run

`SimulationEngine` owns time, data and per-agent state; the decision rule owns
behaviour. With no explicit rule it defaults to `SEURule` built from the
`CouplingConfig` (the validated MVP configuration, including the 75-year
floodproofing lifespan reset).

In [ ]:
config = CouplingConfig()   # DecisionConfig defaults = validated MVP settings
engine = SimulationEngine(ds=ds, config=config)
print(f"residential agents: {engine.n_agents}")

results = engine.run(slr_values, no_seq=NO_SEQ, seed=SEED, track_eu=True)
damage = results["damage_history"]        # (no_seq, n_agents, n_years)
adopted = results["adapted_history"]      # (no_seq, n_agents, n_years)
adoption = results["adoption_fraction"]   # (no_seq, n_years)

## 3. Results — adoption and damages (mean ± 1 s.d. across sequences)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

m, s = adoption.mean(axis=0), adoption.std(axis=0)
ax1.plot(years, m, color="C0", label="SEU adoption (mean)")
ax1.fill_between(years, m - s, m + s, alpha=0.2, color="C0", label="±1 s.d.")
ax1.set_xlabel("Year"); ax1.set_ylabel("Adopted fraction"); ax1.set_ylim(0, 1)
ax1.set_title("Household floodproofing adoption"); ax1.legend()

tot = damage.sum(axis=1) / 1e6            # (no_seq, n_years), M$
md, sd = tot.mean(axis=0), tot.std(axis=0)
ax2.bar(years, md, yerr=sd, capsize=2, color="C1", alpha=0.8)
ax2.set_xlabel("Year"); ax2.set_ylabel("Total damage (M$)")
ax2.set_title("Realised damages under adaptation")

fig.tight_layout()

## 4. Baseline comparison and avoided damages

Run the same event sequences with a never-adapt baseline by using a rule that
always declines (any `DecisionRule` works — here a 100% threshold that is never
reached, since realised damage is capped at `max_pot_dmg`).

In [ ]:
baseline_engine = SimulationEngine(
    ds=ds,
    config=CouplingConfig(decision=DecisionConfig(lifespan_dryproof=None)),
    decision_rule=ThresholdRule(config.decision, damage_threshold=1.01),
)
baseline = baseline_engine.run(slr_values, no_seq=NO_SEQ, seed=SEED)

avoided = np.clip(
    baseline["damage_history"].astype(np.float64) - damage.astype(np.float64),
    0, None,
).sum(axis=1) / 1e6

plt.figure(figsize=(7, 4))
plt.bar(years, avoided.mean(axis=0), yerr=avoided.std(axis=0), capsize=2,
        color="C2", alpha=0.8)
plt.xlabel("Year"); plt.ylabel("Avoided damage (M$)")
plt.title("Damage avoided by SEU-driven floodproofing")
plt.tight_layout()

## 5. Swapping the decision rule (Strategy Pattern)

```python
# Legacy 0.3-threshold behaviour (reproduces ABMSimulator bit-for-bit):
engine = SimulationEngine(ds=ds, config=config,
                          decision_rule=ThresholdRule(config.decision, 0.30))

# Live native DYNAMO-M (parity oracle; needs a DYNAMO-M checkout):
from floodadapt_abm import DynamoLiveRule, DYNAMO_M_AVAILABLE
if DYNAMO_M_AVAILABLE:
    engine = SimulationEngine(ds=ds, config=config,
                              decision_rule=DynamoLiveRule(config.decision))
```

For custom rules, Monte-Carlo uncertainty analysis and the Phase-4b
Mesa-native tick driver, follow the numbered scripts in
[`examples_engine/`](examples_engine/README.md).